# ML exploration: WR next-week PPR (nflverse / ffverse)

Experiments for **next-week PPR** projections using `nflreadpy` + project modules under `src/wr_predictor`.


In [ ]:
# Run this cell first. If you see "Kernel ready" below, the kernel is working.
print("Kernel ready")

In [ ]:
# Project root on sys.path (same pattern as 01_data_exploration)
import os
import sys
from pathlib import Path

cwd = Path(os.getcwd())
PROJECT_ROOT = cwd if (cwd / "src").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root on sys.path:", PROJECT_ROOT)

## 1. Load WR-week data

`build_training_dataset` loads nflverse weekly stats, merges **schedule** context (`spread_line`, `total_line`, `is_home`, `is_dome`, …), builds **lags / rolling** usage (`features.py`), optionally joins **ffopportunity** columns (`merge_ff_opportunity=True`), and adds `next_week_ppr_points`.


In [ ]:
import polars as pl

from src.wr_predictor.dataset_builder import build_training_dataset

TRAINING_SEASONS = list(range(2015, 2025))  # adjust if your nflreadpy release lacks early years

wr_weekly = build_training_dataset(
    seasons=TRAINING_SEASONS,
    min_games_for_player=0,
    merge_ff_opportunity=True,
)

print(wr_weekly.shape)
wr_weekly.head()

## 2. Target and leakage

- **Target**: `next_week_ppr_points` — same-week `ppr_points` shifted **-1** within each `player_id` (sorted by season, week).
- **Features**: lags use `shift(1)`; rolling means apply `shift(1)` before the window so the **current game’s box score is not inside the rolling window** for that row.
- **Never** use the target or any future-week stats as inputs.


In [ ]:
target_col = "next_week_ppr_points"

id_cols = [
    c
    for c in [
        "player_id",
        "player_name",
        "player_display_name",
        "season",
        "week",
        "team",
        "opponent_team",
        "home_away",
    ]
    if c in wr_weekly.columns
]

feature_cols = [c for c in wr_weekly.columns if c not in id_cols and c != target_col]
model_frame = wr_weekly.select(id_cols + feature_cols + [target_col])
model_frame.head()

In [ ]:
# Require non-null rolling-3 features; drop nulls in numeric feature/target columns


def _polars_numeric(schema, col: str) -> bool:
    dt = schema.get(col)
    return bool(dt) and getattr(dt, "is_numeric", lambda: False)()


numeric_cols = [c for c in feature_cols + [target_col] if _polars_numeric(wr_weekly.schema, c)]

roll3 = [c for c in model_frame.columns if c.endswith("_rolling_3")]
if roll3:
    mask = model_frame[roll3[0]].is_not_null()
    for c in roll3[1:]:
        mask = mask & model_frame[c].is_not_null()
    model_frame_clean = model_frame.filter(mask)
else:
    model_frame_clean = model_frame

model_frame_clean = model_frame_clean.drop_nulls(
    subset=[c for c in numeric_cols if c in model_frame_clean.columns]
)
model_frame_clean.shape

## 3. Time-aware train / validation / test split

- **Train**: seasons 2015–2021  
- **Validation**: 2022  
- **Test**: 2023–2024  


In [ ]:
import pandas as pd

pdf = model_frame_clean.to_pandas()
season_col = "season"

train_mask = pdf[season_col].between(2015, 2021)
val_mask = pdf[season_col] == 2022
test_mask = pdf[season_col].between(2023, 2024)

X_cols = [
    c
    for c in feature_cols
    if c in pdf.columns and pd.api.types.is_numeric_dtype(pdf[c])
]
y_col = target_col

X_train, y_train = pdf.loc[train_mask, X_cols], pdf.loc[train_mask, y_col]
X_val, y_val = pdf.loc[val_mask, X_cols], pdf.loc[val_mask, y_col]
X_test, y_test = pdf.loc[test_mask, X_cols], pdf.loc[test_mask, y_col]

X_train.shape, X_val.shape, X_test.shape

## 4. Ridge baseline

Grid over `alpha` using **validation RMSE** (time-ordered val year).


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:

    def root_mean_squared_error(y_true, y_pred):  # type: ignore[misc]
        return float(mean_squared_error(y_true, y_pred, squared=False) ** 0.5)


alphas = [0.1, 1.0, 10.0]
best_alpha = best_val_rmse = best_model = None

for alpha in alphas:
    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha, random_state=0)),
        ]
    )
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_rmse = root_mean_squared_error(y_val, val_pred)
    if best_val_rmse is None or val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_alpha = alpha
        best_model = model

print(f"Best alpha: {best_alpha}, validation RMSE: {best_val_rmse:.3f}")

train_pred = best_model.predict(X_train)
val_pred = best_model.predict(X_val)
test_pred = best_model.predict(X_test)

for split_name, y_true, y_hat in [
    ("Train", y_train, train_pred),
    ("Validation", y_val, val_pred),
    ("Test", y_test, test_pred),
]:
    mae = mean_absolute_error(y_true, y_hat)
    rmse = root_mean_squared_error(y_true, y_hat)
    r2 = r2_score(y_true, y_hat)
    print(f"{split_name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R^2={r2:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual next-week PPR")
plt.ylabel("Predicted (Ridge)")
plt.title("Ridge: predicted vs actual (test)")
plt.tight_layout()
plt.show()

## 5. Neural net baseline (`MLPRegressor`)

Same numeric `X_*` as Ridge. Uses **early stopping** on a holdout fraction of the training split (not the same as the 2022 validation year). For strict temporal validation of the NN, you could instead implement Keras/PyTorch with `X_val`/`y_val`.


In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPRegressor(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                solver="adam",
                alpha=1e-4,
                batch_size=256,
                max_iter=200,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=0,
            ),
        ),
    ]
)

mlp_model.fit(X_train, y_train)
mlp_test_pred = mlp_model.predict(X_test)

print(
    "MLP Test:",
    f"MAE={mean_absolute_error(y_test, mlp_test_pred):.3f}",
    f"RMSE={root_mean_squared_error(y_test, mlp_test_pred):.3f}",
    f"R^2={r2_score(y_test, mlp_test_pred):.3f}",
)

## 6. Interpretability

**Ridge**: standardized coefficient magnitudes.  
**MLP**: quick **permutation importance** on a small test slice (slow if you increase `n_repeats` / sample size).


In [ ]:
ridge = best_model.named_steps["ridge"]
coefs = pd.Series(ridge.coef_, index=X_cols).abs().sort_values(ascending=False)
coefs.head(20)

In [ ]:
from sklearn.inspection import permutation_importance

sample_n = min(800, len(X_test))
X_perm = X_test.sample(sample_n, random_state=0)
y_perm = y_test.loc[X_perm.index]

r = permutation_importance(
    mlp_model,
    X_perm,
    y_perm,
    n_repeats=5,
    random_state=0,
    scoring="r2",
)
imp = pd.Series(r.importances_mean, index=X_cols).sort_values(ascending=False)
imp.head(15)

## 7. ffverse / ffopportunity note

With `merge_ff_opportunity=True`, numeric opportunity columns from `data_loader.load_ff_opportunity` are left-joined on `(player_id, season, week)` and kept in the modeling frame when present. If the join fails (missing keys / empty release), the notebook still runs with usage-only features.


## 8. Projections for a target week

For each WR, take the **last REG game strictly before** `(TARGET_SEASON, TARGET_WEEK)`, require the same non-null `_rolling_3` columns as training, then run `best_model.predict`.


In [ ]:
TARGET_SEASON = 2024
TARGET_WEEK = 6


def projection_rows(df: pl.DataFrame) -> pl.DataFrame:
    prior = df.filter(
        (pl.col("season") < TARGET_SEASON)
        | ((pl.col("season") == TARGET_SEASON) & (pl.col("week") < TARGET_WEEK))
    )
    if "season_type" in prior.columns:
        prior = prior.filter(pl.col("season_type") == "REG")
    prior = prior.sort(["player_id", "season", "week"])
    return prior.unique(subset=["player_id"], keep="last")


proj_pl = projection_rows(wr_weekly)
if roll3:
    m = proj_pl[roll3[0]].is_not_null()
    for c in roll3[1:]:
        m = m & proj_pl[c].is_not_null()
    proj_pl = proj_pl.filter(m)

proj_pdf = proj_pl.to_pandas()
X_proj = proj_pdf[X_cols].astype(float)
proj_pdf = proj_pdf.assign(pred_next_week_ppr_ridge=best_model.predict(X_proj))

display_cols = [c for c in ("player_display_name", "player_name", "team", "season", "week") if c in proj_pdf.columns]
proj_pdf.sort_values("pred_next_week_ppr_ridge", ascending=False)[
    display_cols + ["pred_next_week_ppr_ridge"]
].head(25)

## 9. Summary and next steps

- **Data**: nflverse via `nflreadpy`; schedules for game context; optional ffopportunity via `merge_ff_opportunity`.
- **Target**: `next_week_ppr_points`.
- **Models**: Ridge (strong linear baseline) + `MLPRegressor`.
- **Next**: opponent defensive metrics, injury/participation filters, gradient boosting (`HistGradientBoostingRegressor`, XGBoost), calibration / quantile models for uncertainty.
